# 04 — Temporal Analysis

The knowledge graph is not a snapshot — it's a transition system. When documents carry timestamps, you can walk an entity's history, aggregate corpus-level assertions over time, and replay the graph as it looked on any prior date.

Three design decisions shape how temporal works in the cassette substrate:

1. **Query-time NEXT, not ingest-time.** Other systems materialize NEXT edges at ingest by sorting per-anchor infons by timestamp and writing the edges to disk. We don't — because delta ingests would then need to rewrite edge files, which breaks the cassette substrate's append-only promise. Instead, `store.trajectory(anchor)` reads the `by_anchor` Parquet index and sorts at query time. Cost: a few ms per entity. Benefit: delta ingests stay truly append-only, and the trajectory is always consistent with HEAD.

2. **Every ingest is a snapshot.** Each `store.ingest()` call creates a new `Manifest` snapshot that points at a superset of its parent's cassettes. `store.snapshots()` lists them; `Manifest.load_at(root, snap_id)` replays the store as it looked after that ingest. No special "temporal table" layer — the snapshot chain is the time dimension.

3. **Constraints are aggregates over time, not materialized rows.** `store.constraint(s, p, o)` reads the by_triple index and computes evidence count, polarity balance, time span, and mean confidence on demand. A later retraction of the same triple is visible immediately — no consolidation step to run.

This notebook walks all three.

In [ ]:
import json, tempfile, os
from cognition.cassette import (
    InfonStore, Query, first_seen, last_seen, timeline, next_edges,
)
from cognition.cassette.index import Manifest

SCHEMA = {
    "toyota":    {"type": "actor", "tokens": ["toyota"]},
    "tesla":     {"type": "actor", "tokens": ["tesla"]},
    "ford":      {"type": "actor", "tokens": ["ford"]},
    "bmw":       {"type": "actor", "tokens": ["bmw"]},
    "byd":       {"type": "actor", "tokens": ["byd"]},
    "panasonic": {"type": "actor", "tokens": ["panasonic"]},
    "invest":    {"type": "relation", "tokens": ["invest", "investment"]},
    "launch":    {"type": "relation", "tokens": ["launch", "unveil", "release"]},
    "partner":   {"type": "relation", "tokens": ["partner", "partnership", "collaborate"]},
    "expand":    {"type": "relation", "tokens": ["expand", "expansion", "grow"]},
    "batteries":   {"type": "feature", "tokens": ["battery", "batteries"]},
    "solid_state": {"type": "feature", "tokens": ["solid-state", "solid state"]},
    "ev":          {"type": "feature", "tokens": ["ev", "electric vehicle"]},
    "autonomous":  {"type": "feature", "tokens": ["autonomous", "self-driving"]},
}

tmpdir = tempfile.mkdtemp(prefix="cognition_04_")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

store = InfonStore(os.path.join(tmpdir, "store"), schema_path=schema_path)

## Ingest as a sequence of snapshots

We'll ingest the corpus in three chronological batches, so each batch produces its own snapshot in the manifest chain. Then time-travel to an earlier snapshot shows the store exactly as it was after that batch.

In [ ]:
# Batch 1 — Q1 2024
BATCH_1 = [
    {"id": "d01", "timestamp": "2024-01-15",
     "text": "Toyota invested in solid-state battery research."},
    {"id": "d02", "timestamp": "2024-02-10",
     "text": "Tesla expanded battery production in China."},
    {"id": "d03", "timestamp": "2024-03-20",
     "text": "Ford partnered with BMW on EV development."},
]

# Batch 2 — Q2 2024
BATCH_2 = [
    {"id": "d04", "timestamp": "2024-04-15",
     "text": "Toyota partnered with Panasonic on batteries."},
    {"id": "d05", "timestamp": "2024-05-10",
     "text": "BYD launched an EV in Europe."},
    {"id": "d06", "timestamp": "2024-06-20",
     "text": "Tesla invested in autonomous driving."},
]

# Batch 3 — Q3 + Q4 2024
BATCH_3 = [
    {"id": "d07", "timestamp": "2024-07-15",
     "text": "Toyota expanded EV production in Europe."},
    {"id": "d08", "timestamp": "2024-10-15",
     "text": "Tesla launched self-driving in China."},
    {"id": "d09", "timestamp": "2024-11-10",
     "text": "Toyota launched a solid-state battery vehicle."},
]

store.ingest(BATCH_1); snap_q1 = store.manifest.snapshot_id
store.ingest(BATCH_2); snap_q2 = store.manifest.snapshot_id
store.ingest(BATCH_3); snap_q3 = store.manifest.snapshot_id

print(f"snapshots: {store.snapshots()}")
print(f"  after Q1: {snap_q1}")
print(f"  after Q2: {snap_q2}")
print(f"  after Q3: {snap_q3}  (HEAD)")
print(f"HEAD cassettes: {len(store.manifest.cassettes)}")

## 1. Trajectory — time-ordered sequence per entity

`store.trajectory(anchor)` returns the anchor's hits sorted by timestamp. This is the operation that would have been called "NEXT chain" in a pre-cassette design. We don't materialize the edges; we derive them from the sorted trajectory at query time.

Pass `hydrate=False` to get just the index hits (cheaper); the default hydrates into full `Infon` objects.

In [ ]:
traj = store.trajectory("toyota")
print(f"Toyota trajectory ({len(traj)} hits):")
for inf in traj:
    pol = "\u00ac" if inf.polarity == 0 else " "
    print(f"  {inf.timestamp}  {pol}{inf.subject:<10} {inf.predicate:<10} "
          f"{inf.object:<14}  conf={inf.confidence:.2f}")

## 2. NEXT edges — derived, not stored

`store.next_edges(anchor)` groups the trajectory into consecutive pairs. Each edge carries the two endpoints' timestamps and the gap in days. This is pure index arithmetic — no hydration.

In [ ]:
edges = store.next_edges("toyota")
print(f"Toyota NEXT edges ({len(edges)}):")
for e in edges:
    gap = f"{e.gap_days}d" if e.gap_days is not None else "?"
    print(f"  {e.from_timestamp} \u2192 {e.to_timestamp}   gap={gap}")

## 3. Time-travel — query the store as it was

Every ingest produced a snapshot. `Manifest.load_at(root, snap_id)` rebuilds the manifest at that snapshot; `store.at(snap_id)` returns a read-only view of the store pinned to it. Queries against that view see only cassettes that existed at the snapshot.

This is free because the manifest chain is append-only — no storage overhead, no replay cost.

In [ ]:
def toyota_count_at(snap_id):
    m = Manifest.load_at(store.root, snap_id)
    return len(Query().where(subject="toyota").run(m))

print("Toyota-as-subject infons over time:")
print(f"  after Q1 ({snap_q1[:15]}): {toyota_count_at(snap_q1)}")
print(f"  after Q2 ({snap_q2[:15]}): {toyota_count_at(snap_q2)}")
print(f"  after Q3 ({snap_q3[:15]}): {toyota_count_at(snap_q3)}")

# Run the same query against HEAD vs Q1.
q = Query().where(subject="toyota")
head_hits = q.run(store.manifest)
q1_view = store.at(snap_q1)
q1_hits  = q.run(q1_view.manifest)

print(f"\nSame query against HEAD: {len(head_hits)} hits")
print(f"Same query against Q1  : {len(q1_hits)} hits")
print("HEAD includes everything Q1 had PLUS what landed in Q2 and Q3.")

## 4. Aggregates over time — first_seen, last_seen, timeline, constraint

The index has every timestamp. You don't need to hydrate to answer "when did we first hear about X?" or "which triples are contested?".

In [ ]:
print("first_seen:")
for a in ("toyota", "byd", "panasonic"):
    print(f"  {a:<10} {first_seen(store.manifest, a)}")

print("\nlast_seen:")
for a in ("toyota", "byd", "panasonic"):
    print(f"  {a:<10} {last_seen(store.manifest, a)}")

print("\ntimeline('toyota'):")
for ts, h in timeline(store.manifest, "toyota"):
    pol = "\u00ac" if h.loc.polarity == 0 else " "
    print(f"  {ts}  {pol}{h.loc.subject}/{h.loc.predicate}/{h.loc.object}")

## 5. Constraints — corpus-level evidence over time

`store.constraint(s, p, o)` aggregates every infon matching that triple: how many affirmations, how many refutations, what time span, whether the triple is contested. This is the cassette equivalent of the old `aggregate_constraints` — but computed from the by_triple index on demand, so later retractions update the aggregate automatically with no consolidation step.

Let's show this by adding a contradicting infon in a fourth ingest.

In [ ]:
# Inspect before the retraction lands.
c = store.constraint("toyota", "partner", "panasonic")
print("Before retraction:")
print(f"  evidence:  {c.evidence_count}  "
      f"(aff={c.n_affirmed}, ref={c.n_refuted})")
print(f"  contested: {c.is_contested}")
print(f"  span:      {c.t_min} \u2192 {c.t_max}")
print(f"  balance:   {c.polarity_balance:+.2f}")

# Now append a retraction in a new batch.
store.ingest([{"id": "d10", "timestamp": "2025-03-01",
                "text": "Toyota no longer partnered with Panasonic."}])

c = store.constraint("toyota", "partner", "panasonic")
print("\nAfter retraction landed:")
print(f"  evidence:  {c.evidence_count}  "
      f"(aff={c.n_affirmed}, ref={c.n_refuted})")
print(f"  contested: {c.is_contested}")
print(f"  span:      {c.t_min} \u2192 {c.t_max}  ({c.span_days} days)")
print(f"  balance:   {c.polarity_balance:+.2f}")

## 6. What replaces importance decay

The pre-cassette system had per-infon importance that decayed over time, with a pruning step that soft-deleted stale triples. This mutable approach fought the cassette substrate's append-only guarantee.

What the cassette system does instead:

- **Recency is a property of the query, not the store.** Queries that want recent evidence use `.after(t)` or `.between(a, b)` on the `Query` to scope down.
- **Old snapshots are always available.** No need to prune to free space — cassettes are immutable byte blobs; you can delete cassettes physically if you want (just remove the file and the manifest entry), but the DSL doesn't rely on it.
- **Constraint aggregates give you time-weighted evidence for free.** `c.span_days` tells you how long the triple has persisted; a one-off gets a small span, a durable fact gets a long one.

This is a substrate-level trade: we gave up mutable decay to preserve delta-append and time-travel. The query-time primitives recover everything the decay machinery was trying to approximate.

In [ ]:
# Example: recent Toyota news only.
recent = Query().where(subject="toyota").after("2024-09-01").run(store.manifest)
print(f"Toyota infons after 2024-09-01: {len(recent)}")
for h in recent:
    print(f"  {h.loc.timestamp}  {h.loc.subject}/{h.loc.predicate}/{h.loc.object}")

---

**What you saw:**

1. `trajectory` returns time-ordered hits per entity; `next_edges` derives consecutive pairs. Query-time, not ingest-time.
2. Every ingest creates a snapshot. `Manifest.load_at` or `store.at(snap_id)` replays the graph as it was.
3. `first_seen` / `last_seen` / `timeline` pull extremes and sequences from the index at zero hydration cost.
4. `store.constraint` aggregates a triple's evidence over time; a retraction appended in a later ingest updates the aggregate immediately.
5. The cassette substrate drops mutable importance-decay + pruning in favor of timestamp-bounded queries and immutable snapshots. Same answers, different machinery.

**Next:** [05 — Multi-Domain](05_domains.ipynb) — same cassette substrate, three different domains side-by-side.

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done.")